# Phase 1 — ConvNeXt-Tiny (public notebook)

Derived from the supplied experiment notebook. Run the cells in order to audit the fixed split, train, and evaluate. **Running all cells trains a new model.** The original published results remain in `results/convnext_tiny/`; new runs go to `runs/convnext_tiny/` by default.

## Inputs and environment

- Install compatible Python packages: `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, and `scikit-learn`. The original exact package versions were not recorded here; this is not a pinned environment.
- Set the paths below. Supply the three original `fixed_*_split_public.csv` manifests in `data/`. No new split is generated.
- Supply original images separately in `images/has_arch/` and `images/no_arch/`; the original nested class-folder layout is also supported. Images and trained weights are not embedded in this notebook.
- ImageNet pretrained weights are downloaded by torchvision when absent from its cache. CUDA is recommended; CPU execution is supported but slower.
- Class labels: 0 = No Arch; 1 = Has Arch. Fixed split sizes: 5,235 / 748 / 1,496.

## Preserved experiment

224 x 224 resize and ImageNet normalization; no augmentation; seed 42; batch size 32; AdamW; learning rate and weight decay 0.0001; label smoothing 0.1; up to 20 epochs; early stopping patience 5; minimum validation loss selects the checkpoint. Only `features[7]` and the classifier are trainable.

The audit checks IDs and resolved paths, **not image-content hashes**; it does not establish absence of byte-identical images at different paths. The separate ResNet50 sensitivity analysis addresses exact overlaps.

## Outputs and validation

The training cell writes the checkpoint, training history, configuration, test metrics, and per-image predictions. A new run can differ with hardware or software versions. Original supplied results: accuracy 0.9197860962566845, macro F1 0.917556831228473, best epoch 6, confusion matrix [[565, 54], [66, 811]]. These are reference results, not a new execution of this public notebook.

Public preparation removed Drive discovery, diagnostic cells, stored outputs, and private paths. The executable training and evaluation logic was retained. Syntax and notebook structure were checked; full training was not rerun because the original images are not available here.


In [ ]:
from pathlib import Path

# Set these three paths before running the notebook.
# Start Jupyter from the repository root, or set REPO_ROOT explicitly.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "src":
    REPO_ROOT = REPO_ROOT.parent
DATASET_ROOT = REPO_ROOT / "images"
SPLIT_ROOT = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "runs" / "convnext_tiny"
OUTPUT_ROOT = OUTPUT_DIR

TRAIN_CSV = SPLIT_ROOT / "fixed_train_split_public.csv"
VAL_CSV = SPLIT_ROOT / "fixed_validation_split_public.csv"
TEST_CSV = SPLIT_ROOT / "fixed_test_split_public.csv"


In [ ]:
# ============================================================
# STEP 5 — FINAL Phase-1 split integrity and image-path audit
# READ ONLY
#
# Purpose:
#   1) Lock the exact fixed CSV splits used in the study
#   2) Verify every CSV row resolves to an existing image
#   3) Verify labels/classes/counts
#   4) Confirm no overlap between fixed split memberships
#
# NO TRAINING
# NO RESPLITTING
# NO DATA MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import os

# ------------------------------------------------------------
# FINAL PHASE-1 PATHS — LOCKED
# ------------------------------------------------------------

EXPECTED = {
    "train": {
        "total": 5235,
        "has_arch": 3069,
        "no_arch": 2166,
    },
    "validation": {
        "total": 748,
        "has_arch": 438,
        "no_arch": 310,
    },
    "test": {
        "total": 1496,
        "has_arch": 877,
        "no_arch": 619,
    }
}

# ------------------------------------------------------------
# Resolver copied from the FINAL repository baseline logic
# ------------------------------------------------------------

def resolve_image_path(row):

    relative = Path(str(row["relative_reference"]))
    candidate = DATASET_ROOT / relative

    if candidate.exists():
        return candidate

    class_dir = "has_arch" if int(row["label"]) == 1 else "no_arch"
    filename = str(row["original_filename"])

    candidates = [
        DATASET_ROOT / class_dir / filename,
        DATASET_ROOT / class_dir / class_dir / filename,
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


# ------------------------------------------------------------
# Load exact split files
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

splits = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

required_columns = {
    "sample_id",
    "original_filename",
    "class_name",
    "label",
    "split",
    "relative_reference",
}

print("=" * 90)
print("PHASE 1 — FINAL FIXED-SPLIT INTEGRITY AUDIT")
print("=" * 90)

all_ok = True

resolved_paths = {}

for split_name, df in splits.items():

    print(f"\n[{split_name.upper()}]")

    # ----------------------------------------
    # Column validation
    # ----------------------------------------

    missing_cols = required_columns - set(df.columns)

    if missing_cols:
        print("ERROR — missing columns:", sorted(missing_cols))
        all_ok = False
        continue

    # ----------------------------------------
    # Count validation
    # ----------------------------------------

    total = len(df)
    has_arch = int((df["label"] == 1).sum())
    no_arch = int((df["label"] == 0).sum())

    print(f"Total    : {total}")
    print(f"Has Arch : {has_arch}")
    print(f"No Arch  : {no_arch}")

    expected = EXPECTED[split_name]

    count_ok = (
        total == expected["total"]
        and has_arch == expected["has_arch"]
        and no_arch == expected["no_arch"]
    )

    print(f"Expected counts match : {count_ok}")

    if not count_ok:
        all_ok = False

    # ----------------------------------------
    # Label ↔ class-name consistency
    # ----------------------------------------

    label_class_ok = (
        ((df["label"] == 1) & (df["class_name"] == "Has Arch"))
        |
        ((df["label"] == 0) & (df["class_name"] == "No Arch"))
    ).all()

    print(f"Label/class consistency: {label_class_ok}")

    if not label_class_ok:
        all_ok = False

    # ----------------------------------------
    # Resolve every image
    # ----------------------------------------

    paths = []
    missing_images = []

    for idx, row in df.iterrows():

        path = resolve_image_path(row)

        if path is None:
            missing_images.append(
                {
                    "row": idx,
                    "sample_id": row["sample_id"],
                    "filename": row["original_filename"],
                    "label": row["label"],
                }
            )
        else:
            paths.append(str(path.resolve()))

    resolved_paths[split_name] = paths

    print(f"Resolved images       : {len(paths)}")
    print(f"Missing images        : {len(missing_images)}")

    if missing_images:
        all_ok = False

        print("\nFirst missing examples:")
        for item in missing_images[:10]:
            print(item)


# ------------------------------------------------------------
# Cross-split membership overlap audit
# ------------------------------------------------------------

train_ids = set(train_df["sample_id"].astype(str))
val_ids   = set(val_df["sample_id"].astype(str))
test_ids  = set(test_df["sample_id"].astype(str))

print("\n" + "=" * 90)
print("SAMPLE-ID OVERLAP CHECK")
print("=" * 90)

tv = train_ids & val_ids
tt = train_ids & test_ids
vt = val_ids & test_ids

print(f"Train ∩ Validation : {len(tv)}")
print(f"Train ∩ Test       : {len(tt)}")
print(f"Validation ∩ Test  : {len(vt)}")

if tv or tt or vt:
    all_ok = False


# ------------------------------------------------------------
# Exact resolved-file path overlap
# ------------------------------------------------------------

train_paths = set(resolved_paths.get("train", []))
val_paths   = set(resolved_paths.get("validation", []))
test_paths  = set(resolved_paths.get("test", []))

print("\n" + "=" * 90)
print("EXACT FILE-PATH OVERLAP CHECK")
print("=" * 90)

print(f"Train ∩ Validation : {len(train_paths & val_paths)}")
print(f"Train ∩ Test       : {len(train_paths & test_paths)}")
print(f"Validation ∩ Test  : {len(val_paths & test_paths)}")

if (
    train_paths & val_paths
    or train_paths & test_paths
    or val_paths & test_paths
):
    all_ok = False


# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL AUDIT RESULT")
print("=" * 90)

print(f"Dataset root : {DATASET_ROOT}")
print(f"Split root   : {SPLIT_ROOT}")
print(f"Future output: {OUTPUT_ROOT}")

if all_ok:
    print("\nSTATUS: PASS")
    print("The exact original Phase-1 fixed split is ready for ConvNeXt-Tiny.")
    print("No split was regenerated.")
    print("No image was modified.")
    print("No previous result was modified.")
else:
    print("\nSTATUS: CHECK REQUIRED")
    print("Do NOT start ConvNeXt training yet.")

print("=" * 90)
assert all_ok, "Resolve the audit errors before training."


In [ ]:
# ============================================================
# STEP 8 — FINAL ConvNeXt-Tiny Training + Test Evaluation
# PHASE 1 — Reviewer Revision
#
# CONTROLLED COMPARISON:
# Same split
# Same preprocessing
# Same optimizer
# Same learning rate
# Same weight decay
# Same loss
# Same scheduler
# Same early stopping
# Same gradient clipping
# Same seed
# Same checkpoint criterion
# NO augmentation
#
# ONLY THE CNN ARCHITECTURE IS DIFFERENT
# ============================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)


# ============================================================
# 1. LOCKED PHASE-1 CONFIGURATION
# ============================================================

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 20

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

EARLY_STOPPING_PATIENCE = 5
GRAD_CLIP = 1.0
NUM_WORKERS = 2


# ============================================================
# 2. LOCKED PATHS
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = OUTPUT_DIR / "best_convnext_tiny_baseline.pth"
HISTORY_CSV = OUTPUT_DIR / "training_history.csv"
METRICS_JSON = OUTPUT_DIR / "test_metrics.json"
PREDICTIONS_CSV = OUTPUT_DIR / "test_predictions.csv"
CONFIG_JSON = OUTPUT_DIR / "training_configuration.json"


# ============================================================
# 3. SAME SEED PROCEDURE AS FINAL BASELINES
# ============================================================

def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# 4. SAME DATASET CLASS / PATH RESOLVER
# ============================================================

class SplitDataset(Dataset):

    def __init__(self, csv_path, dataset_root, transform=None):

        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.transform = transform

        required = {
            "original_filename",
            "class_name",
            "label",
            "split",
            "relative_reference",
        }

        missing = required - set(self.df.columns)

        if missing:
            raise ValueError(
                f"Missing required CSV columns: {sorted(missing)}"
            )


    def __len__(self):
        return len(self.df)


    def _resolve_path(self, row):

        relative = Path(str(row["relative_reference"]))
        candidate = self.dataset_root / relative

        if candidate.exists():
            return candidate

        class_dir = (
            "has_arch"
            if int(row["label"]) == 1
            else "no_arch"
        )

        filename = str(row["original_filename"])

        candidates = [
            self.dataset_root / class_dir / filename,
            self.dataset_root / class_dir / class_dir / filename,
        ]

        for path in candidates:

            if path.exists():
                return path

        raise FileNotFoundError(
            f"Image not found for sample "
            f"{row.get('sample_id', '')}: {filename}"
        )


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = self._resolve_path(row)

        image = Image.open(image_path).convert("RGB")

        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ============================================================
# 5. SAME BASELINE PREPROCESSING — NO AUGMENTATION
# ============================================================

transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ============================================================
# 6. EXACT FIXED PHASE-1 DATASETS
# ============================================================

train_ds = SplitDataset(
    TRAIN_CSV,
    DATASET_ROOT,
    transform
)

val_ds = SplitDataset(
    VAL_CSV,
    DATASET_ROOT,
    transform
)

test_ds = SplitDataset(
    TEST_CSV,
    DATASET_ROOT,
    transform
)


assert len(train_ds) == 5235
assert len(val_ds) == 748
assert len(test_ds) == 1496


# Exact class-count verification

assert int((train_ds.df["label"] == 1).sum()) == 3069
assert int((train_ds.df["label"] == 0).sum()) == 2166

assert int((val_ds.df["label"] == 1).sum()) == 438
assert int((val_ds.df["label"] == 0).sum()) == 310

assert int((test_ds.df["label"] == 1).sum()) == 877
assert int((test_ds.df["label"] == 0).sum()) == 619


# ============================================================
# 7. SAME DATALOADER CONFIGURATION
# ============================================================

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)


# ============================================================
# 8. BUILD CONVNEXT-TINY
# ============================================================

def build_model(device):

    model = models.convnext_tiny(
        weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
    )

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Architecture-specific equivalent of final high-level stage
    for param in model.features[7].parameters():
        param.requires_grad = True

    # Replace ImageNet output layer with binary classifier
    in_features = model.classifier[2].in_features

    model.classifier[2] = nn.Linear(
        in_features,
        2
    )

    # Entire classifier trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model.to(device)


# ============================================================
# 9. PARAMETER COUNTER
# ============================================================

def count_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    frozen = total - trainable

    return total, trainable, frozen


# ============================================================
# 10. SAME TRAIN / VALIDATION EPOCH FUNCTION
# ============================================================

def run_epoch(
    model,
    loader,
    criterion,
    device,
    optimizer=None,
    scaler=None
):

    training = optimizer is not None

    model.train(training)

    running_loss = 0.0

    all_labels = []
    all_preds = []

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        use_amp = device.type == "cuda"

        with torch.amp.autocast(
            device_type=device.type,
            enabled=use_amp
        ):

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        if training:

            if use_amp:

                scaler.scale(loss).backward()

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP
                )

                scaler.step(optimizer)

                scaler.update()

            else:

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP
                )

                optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_preds.extend(
            preds.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    epoch_acc = accuracy_score(
        all_labels,
        all_preds
    )

    return epoch_loss, epoch_acc


# ============================================================
# 11. SAME FINAL TEST EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    device
):

    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        probs = torch.softmax(
            logits,
            dim=1
        )

        preds = torch.argmax(
            probs,
            dim=1
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_preds.extend(
            preds.cpu().numpy().tolist()
        )

        all_probs.extend(
            probs.cpu().numpy().tolist()
        )

    labels = np.asarray(all_labels)
    preds = np.asarray(all_preds)
    probs = np.asarray(all_probs)

    metrics = {

        "accuracy": float(
            accuracy_score(
                labels,
                preds
            )
        ),

        "macro_f1": float(
            f1_score(
                labels,
                preds,
                average="macro"
            )
        ),

        "confusion_matrix": confusion_matrix(
            labels,
            preds,
            labels=[0, 1]
        ).tolist(),

        "classification_report":
            classification_report(
                labels,
                preds,
                labels=[0, 1],
                target_names=[
                    "No Arch",
                    "Has Arch"
                ],
                output_dict=True,
                zero_division=0,
            ),
    }

    return (
        metrics,
        labels,
        preds,
        probs
    )


# ============================================================
# 12. DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = build_model(device)

total, trainable, frozen = count_parameters(model)


# Verify exact architecture configuration found in Step 6

assert total == 27_821_666
assert trainable == 14_292_482
assert frozen == 13_529_184


print("=" * 90)
print("PHASE 1 — CONVNEXT-TINY BASELINE TRAINING")
print("=" * 90)

print("Device               :", device)

if torch.cuda.is_available():
    print(
        "GPU                  :",
        torch.cuda.get_device_name(0)
    )

print("Train                :", len(train_ds))
print("Validation           :", len(val_ds))
print("Test                 :", len(test_ds))

print("Total parameters     :", f"{total:,}")
print("Trainable parameters :", f"{trainable:,}")
print("Frozen parameters    :", f"{frozen:,}")

print("Augmentation         : NONE")

print("=" * 90)


# ============================================================
# 13. SAME LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ============================================================
# 14. SAME OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        p
        for p in model.parameters()
        if p.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# 15. SAME LR SCHEDULER
# ============================================================

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )
)


# ============================================================
# 16. SAME MIXED-PRECISION LOGIC
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)


# ============================================================
# 17. SAVE LOCKED CONFIGURATION
# ============================================================

training_configuration = {

    "study_phase":
        "Phase 1",

    "experiment":
        "ConvNeXt-Tiny reviewer revision baseline",

    "architecture":
        "ConvNeXt-Tiny",

    "weights":
        "IMAGENET1K_V1",

    "seed":
        SEED,

    "image_size":
        IMG_SIZE,

    "batch_size":
        BATCH_SIZE,

    "max_epochs":
        MAX_EPOCHS,

    "optimizer":
        "AdamW",

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "loss":
        "CrossEntropyLoss",

    "label_smoothing":
        LABEL_SMOOTHING,

    "scheduler":
        "ReduceLROnPlateau",

    "scheduler_factor":
        0.5,

    "scheduler_patience":
        2,

    "minimum_learning_rate":
        1e-6,

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "gradient_clip":
        GRAD_CLIP,

    "augmentation":
        "none",

    "fine_tuning":
        "features[7] + classifier",

    "checkpoint_selection":
        "minimum validation loss",

    "split_source":
        "fixed deposited CSV files",

    "train_size":
        5235,

    "validation_size":
        748,

    "test_size":
        1496,

    "total_parameters":
        total,

    "trainable_parameters":
        trainable,

    "frozen_parameters":
        frozen,
}


with open(
    CONFIG_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_configuration,
        f,
        indent=4
    )


# ============================================================
# 18. TRAINING LOOP
#    Same checkpoint-selection logic as final baselines
# ============================================================

best_val_loss = float("inf")

epochs_without_improvement = 0

history = []


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # ---------------- TRAIN ----------------

    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        criterion,
        device,
        optimizer=optimizer,
        scaler=scaler,
    )

    # --------------- VALIDATION ------------

    val_loss, val_acc = run_epoch(
        model,
        val_loader,
        criterion,
        device,
        optimizer=None,
        scaler=None,
    )

    # Same scheduler criterion
    scheduler.step(val_loss)

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_acc,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_acc,

        "learning_rate":
            current_lr,
    })


    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.4f} | "
        f"LR {current_lr:.2e}"
    )


    # ========================================================
    # BEST CHECKPOINT = MINIMUM VALIDATION LOSS
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "best_val_loss":
                    best_val_loss,

                "configuration":
                    training_configuration,
            },
            BEST_CHECKPOINT
        )

        print(
            f"  -> Best checkpoint updated "
            f"(Val Loss = {best_val_loss:.4f})"
        )

    else:

        epochs_without_improvement += 1


    # Save history after every epoch
    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_CSV,
        index=False
    )


    # ========================================================
    # SAME EARLY STOPPING RULE
    # ========================================================

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            f"\nEarly stopping at "
            f"epoch {epoch}."
        )

        break


# ============================================================
# 19. LOAD BEST VALIDATION-SELECTED CHECKPOINT
# ============================================================

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_epoch = checkpoint["epoch"]

print("\n" + "=" * 90)

print(
    "Best checkpoint loaded from "
    f"epoch {best_epoch}"
)

print(
    "Best validation loss:",
    f"{checkpoint['best_val_loss']:.6f}"
)

print("=" * 90)


# ============================================================
# 20. FINAL HELD-OUT TEST EVALUATION
#     TEST USED ONLY HERE
# ============================================================

metrics, labels, preds, probs = evaluate(
    model,
    test_loader,
    device
)


print("\n" + "=" * 90)

print(
    "PHASE 1 — CONVNEXT-TINY "
    "FINAL TEST RESULTS"
)

print("=" * 90)

print(
    "Test Accuracy :",
    f"{metrics['accuracy']:.4f}"
)

print(
    "Macro F1      :",
    f"{metrics['macro_f1']:.4f}"
)

print(
    "\nConfusion Matrix "
    "[No Arch, Has Arch]:"
)

print(
    np.asarray(
        metrics["confusion_matrix"]
    )
)


print("\nClassification Report")

print(
    classification_report(
        labels,
        preds,
        labels=[0, 1],
        target_names=[
            "No Arch",
            "Has Arch"
        ],
        digits=4,
        zero_division=0,
    )
)


# ============================================================
# 21. SAVE TEST METRICS
# ============================================================

metrics["best_epoch"] = int(best_epoch)

metrics["best_validation_loss"] = float(
    checkpoint["best_val_loss"]
)


with open(
    METRICS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )


# ============================================================
# 22. SAVE PER-IMAGE TEST PREDICTIONS
# ============================================================

prediction_df = test_ds.df.copy()

prediction_df["true_label"] = labels

prediction_df["predicted_label"] = preds

prediction_df["prob_no_arch"] = (
    probs[:, 0]
)

prediction_df["prob_has_arch"] = (
    probs[:, 1]
)


prediction_df.to_csv(
    PREDICTIONS_CSV,
    index=False
)


# ============================================================
# 23. FINAL OUTPUT CHECK
# ============================================================

print("\n" + "=" * 90)

print("FILES SAVED")

print("=" * 90)

print(
    "Checkpoint :",
    BEST_CHECKPOINT
)

print(
    "History    :",
    HISTORY_CSV
)

print(
    "Metrics    :",
    METRICS_JSON
)

print(
    "Predictions:",
    PREDICTIONS_CSV
)

print(
    "Config     :",
    CONFIG_JSON
)

print("\nSTATUS: COMPLETE")

print("=" * 90)